# Aventix Scarlett Real-Data KGRAG
This notebook uses only the real CSV point-cloud dataset. It never edits source data. Per-well estimates are derived from empty-plate subtraction; gravimetric values are plate-level references only.

## 1. Setup & configuration

In [1]:
from pathlib import Path
import os,re,json,hashlib,datetime,warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError: pass

ROOT = Path.cwd()
possible_roots = [
    ROOT / 'dataset/Point_cloud_dataset',
    ROOT.parent / 'dataset/Point_cloud_dataset'
]

DATASET_DIR = None
for candidate in possible_roots:
    if (candidate / 'manifest.csv').exists():
        DATASET_DIR = candidate
        break
    nested = list(candidate.glob('*/manifest.csv')) if candidate.is_dir() else []
    if nested:
        DATASET_DIR = nested[0].parent
        break

if DATASET_DIR is None:
    DATASET_DIR = ROOT / 'dataset/Point_cloud_dataset'

MANIFEST = DATASET_DIR / 'manifest.csv'
WELLS = DATASET_DIR / 'wells.csv'
SCANS = DATASET_DIR / 'scans'
EMPTY = DATASET_DIR / 'empty_reads'
CONFIG = {'TOLERANCE_PCT': 15.0, 'OUTLIER_SIGMA': 3.0, 'MIN_POINTS': 10, 'RADII_MM': {'AB0796': 4.3, 'EP384': 2.0}, 'CALIBRATION_ID': 'SCARLETT-REAL-UNVERIFIED'}
assert MANIFEST.exists() and WELLS.exists(), f'Missing real dataset under {DATASET_DIR}'
print(f'Dataset path successfully resolved: {DATASET_DIR}')


Dataset path successfully resolved: c:\Users\umair\projects\Aventix_soln\dataset\Point_cloud_dataset


## 2. Dataset audit & CSV ingestion

In [2]:
manifest=pd.read_csv(MANIFEST); 
wells=pd.read_csv(WELLS)
print('Dataset:',DATASET_DIR); print('scans:',len(manifest),'well-center rows:',len(wells))
print(manifest[['file','plate','plate_type','liquid','fill_step','empty_read']].to_string(index=False))
assert len(manifest)==15

def load_points(relative_file, folder):
    path=DATASET_DIR/folder/relative_file if not str(relative_file).startswith(str(folder)) else DATASET_DIR/relative_file
    if not path.exists(): path=DATASET_DIR/relative_file
    data=pd.read_csv(path); required={'well','row','col','x_mm','y_mm','z_mm','sensor'}
    missing=required-data.columns
    if missing: raise ValueError(f'{path}: missing {missing}')
    return data


Dataset: c:\Users\umair\projects\Aventix_soln\dataset\Point_cloud_dataset
scans: 15 well-center rows: 2304
                                     file    plate                                   plate_type liquid  fill_step                       empty_read
   scans/W01_EP384_plateA_water_8.9uL.csv  EP384-A             Eppendorf twin.tec PCR plate 384  water          1  empty_reads/E1_EP384_plateA.csv
  scans/W02_EP384_plateA_water_24.7uL.csv  EP384-A             Eppendorf twin.tec PCR plate 384  water          2  empty_reads/E1_EP384_plateA.csv
  scans/W03_EP384_plateA_water_40.1uL.csv  EP384-A             Eppendorf twin.tec PCR plate 384  water          3  empty_reads/E1_EP384_plateA.csv
 scans/W04_AB0796_plateB_water_42.6uL.csv AB0796-B Thermo Abgene AB-0796 (96-well PCR, skirted)  water          1 empty_reads/E2_AB0796_plateB.csv
 scans/W05_AB0796_plateB_water_52.3uL.csv AB0796-B Thermo Abgene AB-0796 (96-well PCR, skirted)  water          2 empty_reads/E2_AB0796_plateB.csv
scans/W06_A

## 3. Point-cloud preprocessing and dual-sensor fusion

In [3]:
def isolate(points, centre, radius):
    cx,cy=centre
    return points[((points.x_mm-cx)**2+(points.y_mm-cy)**2)<=radius**2].copy()

def clean_z(points):
    if len(points)<3: return points.copy()
    z=points.z_mm.astype(float); lo=z.mean()-CONFIG['OUTLIER_SIGMA']*z.std(); hi=z.mean()+CONFIG['OUTLIER_SIGMA']*z.std()
    return points[(z>=lo)&(z<=hi)].copy()

def well_points(scan, well, radius):
    centre=wells[(wells.scan==scan)&(wells.well==well)]
    if centre.empty: return pd.DataFrame(),None
    pts=isolate(load_points(manifest.loc[manifest.file.str.removesuffix('.csv')==scan,'file'].iloc[0],'').rename(columns={'well':'well_label'}), (centre.iloc[0].centre_x_mm,centre.iloc[0].centre_y_mm), radius)
    # CSV well labels are authoritative; geometry isolation protects against boundary bleed.
    return clean_z(pts),centre.iloc[0]


## 4. Empty-plate subtraction and volume estimation

> **Instrument Physics Note (Thermo Multidrop Combi):**
> In a Thermo Multidrop Combi dispenser, the 8-channel dispensing cassette has 8 nozzles aligned along the **Y-axis (Rows A–H)**.
> The plate steps in the **X-axis (Columns 1–12)**. Therefore, Nozzle 1 fills Row A (A1–A12), Nozzle 2 fills Row B (B1–B12), ..., and Nozzle 8 fills Row H (H1–H12).
> In 384-well plates with a 16-channel cassette, Nozzles 1–16 fill Rows A–P.
> **Dispenser Channel == Well Row** (not column).


In [4]:
SURFACE_TENSION = {'water': 0.0728, 'dmso': 0.0436}
DENSITY_MAP = {'water': 0.998, 'dmso': 1.100}
CONTACT_ANGLE_DEG = {'water': 60.0, 'dmso': 45.0}

def calculate_meniscus_effect(liquid, density_g_per_mL, radius_mm):
    liq = str(liquid).lower()
    gamma = SURFACE_TENSION.get(liq, 0.070)
    rho = float(density_g_per_mL) * 1000.0 if pd.notna(density_g_per_mL) else DENSITY_MAP.get(liq, 1.0) * 1000.0
    theta_deg = CONTACT_ANGLE_DEG.get(liq, 50.0)
    cos_theta = np.cos(np.radians(theta_deg))
    g = 9.81
    r_m = radius_mm * 1e-3
    h_m_mm = (2.0 * gamma * cos_theta / (rho * g * r_m)) * 1e3
    h_m_mm = min(h_m_mm, radius_mm * 0.75)
    v_meniscus_uL = 0.5 * np.pi * (radius_mm ** 2) * h_m_mm
    return h_m_mm, v_meniscus_uL

def estimate_well_volume(fluid_height_mm, plate_type, liquid, density_g_per_mL):
    if not np.isfinite(fluid_height_mm) or fluid_height_mm <= 0:
        return np.nan
    if "384" in str(plate_type):
        r0, k = 0.85, 0.11
    else:
        r0, k = 1.65, 0.21
    r_surface = r0 + k * fluid_height_mm
    h_meniscus, v_meniscus = calculate_meniscus_effect(liquid, density_g_per_mL, r_surface)
    v_body = (np.pi / (3.0 * k)) * ((r0 + k * fluid_height_mm) ** 3 - r0 ** 3)
    total_volume = v_body + 0.3 * v_meniscus
    return float(total_volume)

def extract_fluid_surface_z(pts, ptype):
    rim_max = 10.0 if ptype == "EP384" else 13.2
    s2_pts = pts[(pts.sensor == 2) & (pts.z_mm > 5.0) & (pts.z_mm < rim_max)]
    if len(s2_pts) >= 10:
        return float(s2_pts.z_mm.median())
    s1_pts = pts[(pts.sensor == 1) & (pts.z_mm > 1.8) & (pts.z_mm < rim_max)]
    if len(s1_pts) >= 5:
        return float(s1_pts.z_mm.median())
    valid = pts[pts.z_mm < rim_max]
    return float(valid.z_mm.median()) if len(valid) else np.nan

def extract_floor_z(pts, ptype):
    # Restrict floor extraction to true well bottom (< 1.5mm for 96-well, < 2.0mm for 384)
    # to avoid capturing points on the conical sidewalls
    thresh = 2.0 if ptype == "EP384" else 1.5
    floor_pts = pts[(pts.sensor == 1) & (pts.z_mm < thresh)]
    if len(floor_pts) >= 5:
        return float(floor_pts.z_mm.median())
    valid = pts[pts.z_mm < (thresh + 1.0)]
    return float(valid.z_mm.median()) if len(valid) else (1.25 if ptype == "EP384" else 0.80)

empty_cache = {}
for rel in manifest.empty_read.dropna().unique():
    empty_cache[str(rel)] = pd.read_csv(DATASET_DIR / str(rel))

rows = []
for _, meta in manifest.iterrows():
    scan_id = Path(meta.file).stem
    filled = pd.read_csv(DATASET_DIR / meta.file)
    ptype = "EP384" if "384" in str(meta.plate_type) else "AB0796"
    radius = 1.5 if ptype == "EP384" else 3.0
    liquid = str(meta.liquid).lower()
    density = meta.density_g_per_mL
    empty = empty_cache.get(str(meta.empty_read)) if pd.notna(meta.empty_read) else None
    baseline_est = empty is None
    
    scan_wells = wells[wells.scan == scan_id]
    for _, wr in scan_wells.iterrows():
        fp = clean_z(isolate(filled, (wr.centre_x_mm, wr.centre_y_mm), radius))
        z_surf = extract_fluid_surface_z(fp, ptype)
        if empty is not None:
            ep = clean_z(isolate(empty, (wr.centre_x_mm, wr.centre_y_mm), radius))
            z_floor = extract_floor_z(ep, ptype)
        else:
            z_floor = 1.25 if ptype == "EP384" else 0.80
            
        height = z_surf - z_floor if np.isfinite(z_surf) and np.isfinite(z_floor) else np.nan
        volume = estimate_well_volume(height, ptype, liquid, density)
        h_m, v_m = calculate_meniscus_effect(liquid, density, radius) if np.isfinite(height) else (np.nan, np.nan)
        
        sensors = set(fp.sensor.astype(int)) if len(fp) else set()
        well_row = ord(wr.well[0].upper()) - ord('A') + 1
        well_col = int(re.sub(r'[^0-9]', '', wr.well))
        chan_num = well_row
        max_r = 16 if ptype == "EP384" else 8
        max_c = 24 if ptype == "EP384" else 12
        is_edge = (well_row == 1) or (well_row == max_r) or (well_col == 1) or (well_col == max_c)
        
        rows.append({
            **meta.to_dict(), 'scan_id': scan_id,
            'well': wr.well, 'row': well_row, 'col': well_col,
            'channel': chan_num, 'is_edge': is_edge,
            'centre_x_mm': wr.centre_x_mm, 'centre_y_mm': wr.centre_y_mm,
            'radius_mm': radius, 'point_count': len(fp), 'sensor_count': len(sensors),
            'z_floor_mm': z_floor, 'z_surface_mm': z_surf,
            'fluid_height_mm': height, 'meniscus_height_mm': h_m, 'meniscus_volume_uL': v_m,
            'estimated_uL': volume, 'nominal_uL': wr.nominal_uL,
            'baseline_estimated': baseline_est
        })

measurements = pd.DataFrame(rows)
# Avoid division by zero for un-dispensed wells (nominal_uL == 0)
measurements['deviation_pct'] = np.where(
    measurements.nominal_uL > 0,
    (measurements.estimated_uL - measurements.nominal_uL) / measurements.nominal_uL * 100,
    np.where(measurements.estimated_uL.isna(), 0.0, measurements.estimated_uL)
)
measurements['gravimetric_deviation_pct'] = np.where(
    pd.notna(measurements.gravimetric_mean_uL_per_well) & (measurements.gravimetric_mean_uL_per_well > 0),
    (measurements.estimated_uL - measurements.gravimetric_mean_uL_per_well) / measurements.gravimetric_mean_uL_per_well * 100,
    measurements.deviation_pct
)
measurements.head()
measurements.to_csv("measurements.csv")
print(f"Total measurements computed: {len(measurements)}")
print("\nChannel mapping verification (Thermo Multidrop Combi: Channel == Row):")
sample_wells = ["A1", "A12", "B1", "B12", "C1", "H1", "H12"]
print(measurements[measurements.well.isin(sample_wells)][["plate", "well", "row", "col", "channel", "is_edge"]].drop_duplicates().head(10).to_string(index=False))


Total measurements computed: 2304

Channel mapping verification (Thermo Multidrop Combi: Channel == Row):
   plate well  row  col  channel  is_edge
 EP384-A   A1    1    1        1     True
 EP384-A  A12    1   12        1     True
 EP384-A   B1    2    1        2     True
 EP384-A  B12    2   12        2    False
 EP384-A   C1    3    1        3     True
 EP384-A   H1    8    1        8     True
 EP384-A  H12    8   12        8    False
AB0796-B   A1    1    1        1     True
AB0796-B   B1    2    1        2     True
AB0796-B   C1    3    1        3     True


# The difference in Nominal vs Estimated volumes of the reagent

In [5]:
# Verify channel mapping and edge isolation across plates
print('Wells per channel breakdown (EP384: 16 channels x 24 cols = 384, AB0796: 8 channels x 12 cols = 96):')
print(measurements.groupby(['plate_type', 'channel'])['well'].nunique().unstack(fill_value=0))
print('\nPerimeter Edge Wells vs Interior Wells:')
print(measurements['is_edge'].value_counts())


Wells per channel breakdown (EP384: 16 channels x 24 cols = 384, AB0796: 8 channels x 12 cols = 96):
channel                                       1   2   3   4   5   6   7   8   \
plate_type                                                                     
Eppendorf twin.tec PCR plate 384              24  24  24  24  24  24  24  24   
Thermo Abgene AB-0796 (96-well PCR, skirted)  12  12  12  12  12  12  12  12   

channel                                       9   10  11  12  13  14  15  16  
plate_type                                                                    
Eppendorf twin.tec PCR plate 384              24  24  24  24  24  24  24  24  
Thermo Abgene AB-0796 (96-well PCR, skirted)   0   0   0   0   0   0   0   0  

Perimeter Edge Wells vs Interior Wells:
is_edge
False    1644
True      660
Name: count, dtype: int64


## 5. Deterministic verification

> **Verification Architecture:**
> - **RULE_1 (Gravimetric Deviation):** Statistical outlier relative to plate gravimetric scale mean (> 15%).
> - **RULE_2 (Systematic Channel Bias):** Nozzle bias evaluated relative to the interior scan baseline on interior columns (excluding perimeter edge artifacts).
> - **RULE_5 (Perimeter Edge Effect):** Perimeter wells (Row A, Row H, Col 1, Col 12) affected by optical reflection and meniscus boundary effects.
> - **RULE_3, 4, 6, 7, 8, 9:** Bubble misreads, partial scans, baseline estimations, overfill, step regression, and missing wells.


In [6]:
def verify_dataset(df):
    df = df.copy()
    df['rule_id'] = ''
    df['fault_reason'] = ''
    df['status'] = 'PASS'
    
    # 1. Per-Well Base Rules
    for idx, r in df.iterrows():
        reasons = []
        ptype = "EP384" if "384" in str(r.plate_type) else "AB0796"
        max_depth = 10.5 if ptype == "EP384" else 14.5
        
        # RULE_9: Missing well points / unphysical height
        if pd.isna(r.estimated_uL) or r.point_count == 0:
            reasons.append(('RULE_9', 'MISSING_WELL', 'FAIL'))
            
        # RULE_7: Overfill beyond well depth
        if pd.notna(r.fluid_height_mm) and r.fluid_height_mm > max_depth:
            reasons.append(('RULE_7', 'OVERFILL', 'FAIL'))
            
        # RULE_1: Statistical Gravimetric Anomaly (relative to plate gravimetric mean)
        if pd.notna(r.gravimetric_deviation_pct) and abs(r.gravimetric_deviation_pct) > CONFIG['TOLERANCE_PCT']:
            reasons.append(('RULE_1', 'GRAVIMETRIC_DEVIATION', 'FAIL'))
            
        # RULE_3: Low point count / bubble misread
        if r.point_count < CONFIG['MIN_POINTS']:
            reasons.append(('RULE_3', 'BUBBLE_OR_MISREAD', 'WARNING'))
            
        # RULE_4: Single sensor partial scan
        if r.sensor_count < 2 and r.point_count > 0:
            reasons.append(('RULE_4', 'PARTIAL_SCAN', 'WARNING'))
            
        # RULE_6: Baseline estimated (no matching empty read)
        if r.baseline_estimated:
            reasons.append(('RULE_6', 'BASELINE_ESTIMATED', 'WARNING'))
            
        # RULE_5: Edge effect (perimeter well optical / meniscus artifact)
        if r.get('is_edge', False) and pd.notna(r.gravimetric_deviation_pct) and abs(r.gravimetric_deviation_pct) > CONFIG['TOLERANCE_PCT']:
            reasons.append(('RULE_5', 'EDGE_EFFECT', 'WARNING'))
            
        if reasons:
            df.loc[idx, 'fault_reason'] = '; '.join(x[1] for x in reasons)
            df.loc[idx, 'rule_id'] = ';'.join(x[0] for x in reasons)
            df.loc[idx, 'status'] = 'FAIL' if any(x[2] == 'FAIL' for x in reasons) else 'WARNING'
            
    # 2. RULE_8: Step Regression Check (Cumulative Plates A, B, E)
    for p in ['EP384-A', 'AB0796-B', 'AB0796-E']:
        p_mask = df.plate == p
        p_wells = df[p_mask].sort_values(['well', 'fill_step'])
        for well_name, w_grp in p_wells.groupby('well'):
            vols = w_grp.estimated_uL.values
            indices = w_grp.index.values
            for i in range(1, len(vols)):
                if pd.notna(vols[i]) and pd.notna(vols[i-1]) and (vols[i] < vols[i-1] - 1.0):
                    idx_curr = indices[i]
                    curr_rules = df.loc[idx_curr, 'rule_id']
                    df.loc[idx_curr, 'rule_id'] = (curr_rules + ';RULE_8').strip(';')
                    df.loc[idx_curr, 'fault_reason'] = (df.loc[idx_curr, 'fault_reason'] + '; STEP_REGRESSION').strip(';')
                    df.loc[idx_curr, 'status'] = 'FAIL'

    # 3. RULE_2: Systematic Channel Bias Check (relative to interior scan baseline)
    for scan_id, s_grp in df.groupby('scan_id'):
        ptype = "EP384" if "384" in str(s_grp.plate_type.iloc[0]) else "AB0796"
        max_c = 24 if ptype == "EP384" else 12
        valid = s_grp[s_grp.nominal_uL > 0]
        if valid.empty: continue
        
        interior = valid[~valid.is_edge] if 'is_edge' in valid else valid
        scan_baseline = interior['gravimetric_deviation_pct'].median() if len(interior) >= 10 else valid['gravimetric_deviation_pct'].median()
        
        for ch, ch_grp in valid.groupby('channel'):
            ch_test = ch_grp[(ch_grp.col > 1) & (ch_grp.col < max_c)]
            if len(ch_test) < 4: ch_test = ch_grp
            rel_devs = ch_test['gravimetric_deviation_pct'] - scan_baseline
            bad = rel_devs[rel_devs.abs() > 10.0]
            if len(bad) >= 4 and np.sign(bad).nunique() == 1 and abs(rel_devs.median()) > 10.0:
                for bad_idx in bad.index:
                    curr_rules = df.loc[bad_idx, 'rule_id']
                    if 'RULE_2' not in curr_rules:
                        df.loc[bad_idx, 'rule_id'] = (curr_rules + ';RULE_2').strip(';')
                        df.loc[bad_idx, 'fault_reason'] = (df.loc[bad_idx, 'fault_reason'] + '; CHANNEL_BIAS').strip(';')
                        
    return df

measurements = verify_dataset(measurements)
print("Verification complete. Status value counts:")
print(measurements['status'].value_counts())
print("\nTop rule combinations:")
print(measurements['rule_id'].value_counts().head(8))


Verification complete. Status value counts:
status
WARNING    951
FAIL       734
PASS       619
Name: count, dtype: int64

Top rule combinations:
rule_id
RULE_4                         820
                               508
RULE_1;RULE_4                  306
RULE_1                         145
RULE_2                         111
RULE_1;RULE_4;RULE_5           104
RULE_1;RULE_4;RULE_5;RULE_2     76
RULE_4;RULE_6                   56
Name: count, dtype: int64


## 6. Knowledge graph construction

In [9]:
G = nx.MultiDiGraph()
G.add_node('Lab:Aventix', type='Lab', name='Aventix Precision Lab')
G.add_node('Instrument:SCARLETT', type='Instrument', instrument_type='optical_scanner', manufacturer='Aventix')
G.add_node('Instrument:Combi', type='Instrument', instrument_type='liquid_dispenser', model='Thermo Multidrop Combi')

# 1. Liquids from measurements
for liquid_name, grp in measurements.groupby('liquid'):
    density = grp['density_g_per_mL'].dropna().iloc[0] if 'density_g_per_mL' in grp else 1.0
    gamma = SURFACE_TENSION.get(str(liquid_name).lower(), 0.070)
    G.add_node(f"Liquid:{liquid_name}", type='Liquid', name=str(liquid_name), density_g_per_mL=float(density), surface_tension_N_m=float(gamma))

# 2. Plates & Empty Read baselines
for plate_id, grp in measurements.groupby('plate'):
    first = grp.iloc[0]
    fmt = 'EP384' if '384' in str(first['plate_type']) else 'AB0796'
    plate_node = f"Plate:{plate_id}"
    G.add_node(plate_node, type='Plate', plate_id=plate_id, plate_type=fmt, wells_count=int(first['wells']))
    if pd.notna(first['empty_read']):
        empty_id = f"EmptyRead:{Path(str(first['empty_read'])).stem}"
        G.add_node(empty_id, type='EmptyRead', file=str(first['empty_read']))
        G.add_edge(plate_node, empty_id, relation='HAS_BASELINE_SCAN')

# 3. Experimental Runs with CDE (Comparability Determination Engine) Fingerprint
for scan_id, grp in measurements.groupby('scan_id'):
    first = grp.iloc[0]
    run_node = f"Run:{scan_id}"
    plate_node = f"Plate:{first['plate']}"
    liquid_node = f"Liquid:{first['liquid']}"
    fmt = 'EP384' if '384' in str(first['plate_type']) else 'AB0796'
    
    # CDE comparability fingerprint
    layout_type = 'ladder' if 'ladder' in str(first.get('layout', '')).lower() else 'uniform'
    fp_text = f"{fmt}|{first['liquid']}|{layout_type}|{CONFIG['CALIBRATION_ID']}"
    fingerprint = hashlib.sha256(fp_text.encode('utf-8')).hexdigest()[:12]
    
    G.add_node(run_node, type='Run', scan_id=scan_id, file=first['file'], fill_step=int(first['fill_step']),
               plate_type=fmt, liquid=str(first['liquid']), layout=layout_type,
               fingerprint=fingerprint, scan_start_utc=str(first['scan_start_utc']))
    G.add_edge('Lab:Aventix', run_node, relation='CONDUCTED_RUN')
    G.add_edge(run_node, plate_node, relation='TARGETED_PLATE')
    G.add_edge(run_node, liquid_node, relation='USED_LIQUID')
    G.add_edge(run_node, 'Instrument:Combi', relation='DISPENSED_BY')

# 4. Wells, Measurements, and Dispenser Channels
wells_added = set()
for _, r in measurements.iterrows():
    plate_node = f"Plate:{r['plate']}"
    well_node = f"Well:{r['plate']}:{r['well']}"
    meas_node = f"Measurement:{r['plate']}:{r['scan_id']}:{r['well']}"
    chan_node = f"DispenserChannel:{r['channel']}"
    
    if well_node not in wells_added:
        wells_added.add(well_node)
        G.add_node(well_node, type='Well', plate_id=r['plate'], well_id=r['well'],
                   row=int(r['row']), col=int(r['col']),
                   centre_x_mm=float(r['centre_x_mm']), centre_y_mm=float(r['centre_y_mm']))
        G.add_edge(plate_node, well_node, relation='HAS_WELL')
        
    G.add_node(meas_node, type='Measurement',
               scan_id=r['scan_id'], well_id=r['well'], plate_id=r['plate'], source_file=r['file'],
               estimated_uL=float(r['estimated_uL']) if pd.notna(r['estimated_uL']) else np.nan,
               nominal_uL=float(r['nominal_uL']) if pd.notna(r['nominal_uL']) else np.nan,
               deviation_pct=float(r['deviation_pct']) if pd.notna(r['deviation_pct']) else 0.0,
               gravimetric_deviation_pct=float(r['gravimetric_deviation_pct']) if pd.notna(r['gravimetric_deviation_pct']) else 0.0,
               status=str(r['status']), fault_reason=str(r['fault_reason']), rule_id=str(r['rule_id']),
               fluid_height_mm=float(r['fluid_height_mm']) if pd.notna(r['fluid_height_mm']) else np.nan,
               channel=int(r['channel']),
               point_count=int(r['point_count']), sensor_count=int(r['sensor_count']),
               baseline_estimated=bool(r['baseline_estimated']))
    G.add_edge(well_node, meas_node, relation='HAS_MEASUREMENT')
    G.add_edge(meas_node, 'Instrument:SCARLETT', relation='MEASURED_BY')
    
    G.add_node(chan_node, type='DispenserChannel', channel_number=int(r['channel']), instrument='Instrument:Combi')
    G.add_edge(meas_node, chan_node, relation='DISPENSED_BY_CHANNEL')

def are_runs_comparable(run_a, run_b):
    """CDE function: evaluates graph node fingerprints and identifies discrepancies."""
    if not run_a.startswith('Run:'): run_a = f"Run:{run_a}"
    if not run_b.startswith('Run:'): run_b = f"Run:{run_b}"
    if run_a not in G or run_b not in G:
        return False, "One or both runs not found in graph."
    fp_a = G.nodes[run_a].get('fingerprint')
    fp_b = G.nodes[run_b].get('fingerprint')
    if fp_a == fp_b:
        return True, f"Matching CDE fingerprint [{fp_a}]: runs share plate format, liquid, and protocol."
    diffs = []
    for k in ['plate_type', 'liquid', 'layout']:
        va = G.nodes[run_a].get(k)
        vb = G.nodes[run_b].get(k)
        if va != vb: diffs.append(f"{k} differs ('{va}' vs '{vb}')")
    return False, f"Incompatible configurations (fingerprint {fp_a} vs {fp_b}): {'; '.join(diffs)}."

print(f"Knowledge Graph successfully built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Knowledge Graph successfully built: 3410 nodes, 8034 edges


## 7. Grounded RAG query layer

In [10]:
def retrieve(question):
    q = question.lower()
    
    plate_hits = set()
    for p in manifest.plate.unique():
        if p.lower() in q:
            plate_hits.add(p)
    short_matches = re.findall(r'\bplate\s*[-_]?\s*([a-h]|ep384[-_]?a?)\b', q)
    for m in short_matches:
        if m.startswith('ep'): plate_hits.add('EP384-A')
        else:
            for p in manifest.plate.unique():
                if p.endswith(m.upper()): plate_hits.add(p)
                
    well_hits = [w.upper() for w in re.findall(r'\b([a-p]\d{1,2})\b', q)]
    chan_matches = re.findall(r'\bchannel\s*(\d+)\b', q)
    target_channels = [int(c) for c in chan_matches]
    
    results = []
    candidate_plates = plate_hits if plate_hits else manifest.plate.unique()
    
    # Traverse through graph edges
    for p_id in candidate_plates:
        plate_node = f"Plate:{p_id}"
        if plate_node not in G: continue
        
        # Step 1: Plate -> HAS_WELL -> Well
        for _, well_node, edata in G.out_edges(plate_node, data=True):
            if edata.get('relation') != 'HAS_WELL': continue
            well_attrs = G.nodes[well_node]
            w_id = well_attrs.get('well_id')
            if well_hits and w_id not in well_hits:
                continue
                
            # Step 2: Well -> HAS_MEASUREMENT -> Measurement
            for _, meas_node, m_edata in G.out_edges(well_node, data=True):
                if m_edata.get('relation') != 'HAS_MEASUREMENT': continue
                meas_attrs = G.nodes[meas_node]
                
                # Step 3: Measurement -> DISPENSED_BY_CHANNEL -> Channel
                chan_num = None
                for _, ch_node, c_edata in G.out_edges(meas_node, data=True):
                    if c_edata.get('relation') == 'DISPENSED_BY_CHANNEL':
                        chan_num = G.nodes[ch_node].get('channel_number')
                        
                if target_channels and chan_num not in target_channels:
                    continue
                    
                if any(term in q for term in ['failed', 'fault', 'anomal', 'issue', 'bias']) and meas_attrs.get('status') not in {'FAIL', 'WARNING', 'UNVERIFIED'}:
                    continue
                    
                results.append({
                    'measurement_id': meas_node,
                    'well_id': w_id,
                    'plate_id': p_id,
                    'channel': chan_num,
                    **meas_attrs
                })
                
    return results[:50]

def verify_llm_grounding(llm_response, facts):
    fact_ids = {f['measurement_id'] for f in facts}
    cited_ids = re.findall(r'Measurement:[A-Za-z0-9_\-\.:]+', llm_response)
    for cid in cited_ids:
        if not any(cid in fid or fid in cid for fid in fact_ids):
            return False, f"LLM fabricated Measurement ID: {cid}"
            
    cited_decimals = [float(x) for x in re.findall(r'[-+]?\d+\.\d+', llm_response)]
    ground_truth_decimals = []
    for f in facts:
        for k in ['estimated_uL', 'nominal_uL', 'deviation_pct', 'gravimetric_deviation_pct']:
            val = f.get(k)
            if pd.notna(val):
                ground_truth_decimals.append(float(val))
                
    for c_val in cited_decimals:
        matches = any(abs(c_val - g_val) < 0.2 or abs(abs(c_val) - abs(g_val)) < 0.2 for g_val in ground_truth_decimals)
        if not matches and c_val > 1.0:
            meta_match = any(str(round(c_val, 1)) in f['measurement_id'] for f in facts)
            if not meta_match:
                return False, f"LLM cited unverified numeric value: {c_val}"
                
    return True, "All cited figures and IDs verified against graph facts."

def ask(question, use_llm=True):
    q = question.lower()
    
    # 1. CDE Comparability Query Handling
    if "comparable" in q or "compare" in q:
        runs_found = [Path(f).stem for f in manifest.file if Path(f).stem.lower() in q]
        plates_found = [p for p in manifest.plate.unique() if p.lower() in q]
        if len(runs_found) >= 2:
            comp, reason = are_runs_comparable(runs_found[0], runs_found[1])
            return {'answer': f"CDE Comparability Assessment: {comp}. {reason}", 'evidence_pack': [], 'grounding_verified': True}
        elif len(plates_found) >= 2:
            run_a = measurements[measurements.plate == plates_found[0]].scan_id.iloc[0]
            run_b = measurements[measurements.plate == plates_found[1]].scan_id.iloc[0]
            comp, reason = are_runs_comparable(run_a, run_b)
            return {'answer': f"CDE Comparability Assessment for {plates_found[0]} vs {plates_found[1]}: {comp}. {reason}", 'evidence_pack': [], 'grounding_verified': True}
            
    # 2. Subgraph Edge Traversal Retrieval
    facts = retrieve(question)
    if not facts:
        return {'answer': 'No verified data found in the graph for this query.', 'evidence_pack': [], 'grounding_verified': True}
        
    # 3. LLM Generation with Hallucination Guard
    if use_llm and os.getenv('GROQ_API_KEY'):
        try:
            from groq import Groq
            context = '\n'.join(
                f"- {f['measurement_id']}: Well {f.get('well_id')} on Plate {f.get('plate_id')} (Scan {f.get('scan_id')}). Status={f.get('status')}, Est={f.get('estimated_uL', np.nan):.2f} uL, Nom={f.get('nominal_uL', np.nan):.2f} uL, GravDev={f.get('gravimetric_deviation_pct', 0.0):.1f}%. Rules=[{f.get('rule_id', 'none')}]. Fault={f.get('fault_reason', 'none')}."
                for f in facts[:20]
            )
            p = f"Answer in 1-3 concise natural language sentences using only these graph facts. Cite Measurement IDs. Do not invent causes or numbers.\nQuestion: {question}\nFacts:\n{context}"
            model_name = os.getenv('MODEL', 'openai/gpt-oss-120b')
            client = Groq(api_key=os.environ['GROQ_API_KEY'])
            raw_answer = client.chat.completions.create(
                model=model_name,
                temperature=0.1,
                messages=[
                    {'role': 'system', 'content': 'You are a grounded laboratory QC consultant.'},
                    {'role': 'user', 'content': p}
                ]
            ).choices[0].message.content.strip()
            
            # Grounding Post-Check
            is_grounded, check_msg = verify_llm_grounding(raw_answer, facts)
            if not is_grounded:
                raw_answer = f"[Grounding Warning: {check_msg}]\n" + raw_answer
                
            return {'answer': raw_answer, 'evidence_pack': facts, 'grounding_verified': is_grounded}
        except Exception as e:
            print(f"[Notice: LLM call fallback due to: {e}]")
            
    # 4. Deterministic Grounded Synthesis (Fallback)
    lines = []
    lines.append(f"Grounded Findings ({len(facts)} measurement{'s' if len(facts) > 1 else ''} retrieved via graph traversal):")
    for f in facts[:10]:
        status = f.get('status', 'UNKNOWN')
        well = f.get('well_id', 'N/A')
        plate = f.get('plate_id', 'N/A')
        scan = f.get('scan_id', 'N/A')
        est = f.get('estimated_uL', np.nan)
        nom = f.get('nominal_uL', np.nan)
        dev = f.get('gravimetric_deviation_pct', 0.0)
        reasons = f.get('fault_reason', '').strip() or 'None'
        rules = f.get('rule_id', '').strip() or 'None'
        vol_info = f"Measured: {est:.2f} µL (Nominal: {nom:.2f} µL, GravDev: {dev:+.1f}%)" if pd.notna(est) else f"Measured: {est}"
        lines.append(f"• Well {well} ({plate}, Scan {scan}): Status {status} | {vol_info} | Rules: [{rules}] | Reasons: {reasons}")
    if len(facts) > 10:
        lines.append(f"... and {len(facts) - 10} more measurements.")
    return {'answer': '\n'.join(lines), 'evidence_pack': facts, 'grounding_verified': True}


## 8. Visualisation & validation

In [11]:
summary_df = measurements.groupby('plate').agg(
    scans=('scan_id', 'nunique'),
    wells=('well', 'count'),
    mean_uL=('estimated_uL', 'mean'),
    mean_abs_dev_pct=('gravimetric_deviation_pct', lambda x: x.abs().mean())
)
print("Plate-Level Summary (vs Gravimetric Mean):")
print(summary_df.round(2))
print()
print("Verification Status Breakdown:")
print(measurements['status'].value_counts())
assert set(measurements.status).issubset({'PASS', 'FAIL', 'WARNING', 'UNVERIFIED'})
assert measurements[measurements.baseline_estimated].shape[0] > 0

print("\nChannel-Level Diagnostic Breakdown on AB0796-B (Nozzles 1-8 / Rows A-H):")
ch_summary = measurements[measurements.plate == "AB0796-B"].groupby(["channel", "row"]).agg(
    mean_dev=("gravimetric_deviation_pct", "mean"),
    rule2_bias=("rule_id", lambda x: x.str.contains("RULE_2").sum()),
    rule5_edge=("rule_id", lambda x: x.str.contains("RULE_5").sum())
).reset_index()
print(ch_summary.to_string(index=False))


Plate-Level Summary (vs Gravimetric Mean):
          scans  wells  mean_uL  mean_abs_dev_pct
plate                                            
AB0796-B      4    384   145.03             26.14
AB0796-C      1     96    87.63             31.39
AB0796-D      1     96    51.14             54.12
AB0796-E      3    288   122.38             25.55
AB0796-F      1     96   102.19             14.87
AB0796-G      1     96    87.69             19.79
AB0796-H      1     96    49.77             33.84
EP384-A       3   1152    24.76              5.22

Verification Status Breakdown:
status
WARNING    951
FAIL       734
PASS       619
Name: count, dtype: int64

Channel-Level Diagnostic Breakdown on AB0796-B (Nozzles 1-8 / Rows A-H):
 channel  row  mean_dev  rule2_bias  rule5_edge
       1    1 21.563044          30          24
       2    2  1.572564           0           4
       3    3 11.395447          10           4
       4    4 -6.139037           0           4
       5    5 -1.960624          

## 9. Final real-data demo

In [12]:
print("=== Demo 1: Grounded Single-Well Query via Graph Traversal ===")
result1 = ask('Why was well B3 faulty on plate AB0796-B?', use_llm=True)
print("Answer:", result1['answer'])
print("Grounding Verified:", result1.get('grounding_verified', True))
print("Evidence Count:", len(result1['evidence_pack']))

print("\n=== Demo 2: CDE Comparability Determination Engine ===")
result2 = ask('Are plate EP384-A and plate AB0796-B comparable?')
print(result2['answer'])

print("\n=== Demo 3: Systematic Dispenser Channel Bias Query ===")
result3 = ask('Which wells failed due to channel bias on plate AB0796-B?')
print(result3['answer'])


=== Demo 1: Grounded Single-Well Query via Graph Traversal ===
Answer: Well B3 on plate AB0796‑B was flagged because its gravimetric measurements deviated markedly from the nominal volumes: the 42.6 µL and 52.3 µL scans gave negative gravimetric deviations (‑3.3 % and ‑12.0 %) and were marked WARNING (Measurement AB0796‑B:W04_AB0796_plateB_water_42.6uL:B3; AB0796‑B:W05_AB0796_plateB_water_52.3uL:B3), and the 181.8 µL and 311.1 µL scans showed larger negative deviations (‑28.8 % and ‑16.2 %) and were marked FAIL (Measurement AB0796‑B:W06_AB0796_plateB_water_181.8uL:B3; AB0796‑B:W07_AB0796_plateB_water_311.1uL:B3).
Grounding Verified: True
Evidence Count: 4

=== Demo 2: CDE Comparability Determination Engine ===
CDE Comparability Assessment for EP384-A vs AB0796-B: False. Incompatible configurations (fingerprint 5f5a4584ecfe vs a9d92713db7d): plate_type differs ('EP384' vs 'AB0796').

=== Demo 3: Systematic Dispenser Channel Bias Query ===
The wells that failed on plate AB0796‑B with a C